# Full end-to-end Colab pipeline (verbatim)

This notebook is a **faithful, runnable reconstruction** of the original Google Colab export. Every code line is unchanged; only the cell boundaries were rebuilt from the `CELL n` markers, because the `.py` export had flattened them.

- The untouched original file is committed as [`pipeline_full_colab.py`](../pipeline_full_colab.py) (byte-identical, same SHA-256).
- This notebook exists only so the same script can actually **run** again: `!pip install` lines are invalid Python outside a notebook.
- It expects you to upload `Base_Scenario_with_OCE.csv` and `2050_Scenario_with_OCE.csv` when prompted.


In [ ]:
# -*- coding: utf-8 -*-
"""A data-driven multi-objective optimization of building energy consumption, indoor discomfort, and CO2 emissions under future climate change and grid decarbonization.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1Y2BpTS5j-2gWfcuNJagVvjrq-mACz8bv
"""

In [ ]:
!pip install -q catboost optuna shap pymoo

import io
import numpy as np
import pandas as pd
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 2:
    raise ValueError(f"Expected 2 CSV files, but got {len(uploaded)}. Re-run the cell and select both files.")

file_names = list(uploaded.keys())
df1 = pd.read_csv(io.BytesIO(uploaded[file_names[0]]))
df2 = pd.read_csv(io.BytesIO(uploaded[file_names[1]]))

print(f"File 1: {file_names[0]} -> shape {df1.shape}")
print(f"File 2: {file_names[1]} -> shape {df2.shape}")
display(df1.head())
display(df2.head())

def clean_cols(df):
    df = df.copy()
    df.columns = (df.columns.str.strip()
                            .str.replace(" ", "_", regex=False)
                            .str.replace("-", "_", regex=False))
    return df

df1 = clean_cols(df1)
df2 = clean_cols(df2)

def get_scenario(df):
    if any("Baseline" in c for c in df.columns):
        return "Baseline"
    if any("Mid_future" in c for c in df.columns):
        return "Mid_future"
    raise ValueError("Cannot detect scenario from column names.")

scen = {get_scenario(df1): df1, get_scenario(df2): df2}
assert set(scen) == {"Baseline", "Mid_future"}, "Both scenarios must be different."

targets = ["EUI", "IDD", "OCI"]
for name, df in scen.items():
    df.rename(columns={f"{t}_{name}": t for t in targets}, inplace=True)

features = [c for c in scen["Baseline"].columns if c not in targets]
print("Input features :", features)
print("Target outputs :", targets)

base, future = scen["Baseline"], scen["Mid_future"]
print(f"\nSame input rows in both files: {base[features].equals(future[features])}")
for name, df in scen.items():
    print(f"{name}: missing = {df.isna().sum().sum()}, duplicate rows = {df.duplicated().sum()}")

print("\nMean targets per scenario:")
print(pd.DataFrame({n: d[targets].mean() for n, d in scen.items()}).round(3))

X = base[features].copy()
y_base   = base[targets].copy()
y_future = future[targets].copy()

print(f"\nX shape: {X.shape} | y_base: {y_base.shape} | y_future: {y_future.shape}")
display(X.describe().T.round(3))
display(pd.concat({"Baseline": y_base.describe().T,
                   "Mid_future": y_future.describe().T}, axis=0).round(3))

In [ ]:
!pip install -q xgboost bayesian-optimization catboost
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch
from bayes_opt import BayesianOptimization
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

SEED = 42
N_INIT, N_ITER = 10, 30          # use 5, 10 for a quick test
KAPPA = 2.576
R2_DEC = 4
TRAIN_C, TEST_C = "#9b59b6", "#2ecc71"
FIG_DPI = 600                    # use 300 for a quick test

mpl.rcParams.update({"font.family": "serif",
                     "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
                     "mathtext.fontset": "stix", "axes.unicode_minus": False})

idx_train, idx_test = train_test_split(X.index, test_size=0.2, random_state=SEED)
X_train, X_test = X.loc[idx_train], X.loc[idx_test]
Y = {"Baseline": y_base, "Mid_future": y_future}
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

I = lambda v: int(round(float(v)))

def build_cat(p):
    return CatBoostRegressor(iterations=I(p["iterations"]), depth=I(p["depth"]),
                             learning_rate=float(p["learning_rate"]), l2_leaf_reg=float(p["l2_leaf_reg"]),
                             random_strength=float(p["random_strength"]), random_seed=SEED,
                             verbose=0, thread_count=-1, allow_writing_files=False)

def build_xgb(p):
    return XGBRegressor(n_estimators=I(p["n_estimators"]), max_depth=I(p["max_depth"]),
                        learning_rate=float(p["learning_rate"]), subsample=float(p["subsample"]),
                        colsample_bytree=float(p["colsample_bytree"]), reg_lambda=float(p["reg_lambda"]),
                        random_state=SEED, n_jobs=-1, verbosity=0)

def build_rf(p):
    return RandomForestRegressor(n_estimators=I(p["n_estimators"]), max_depth=I(p["max_depth"]),
                                 min_samples_split=I(p["min_samples_split"]),
                                 min_samples_leaf=I(p["min_samples_leaf"]),
                                 max_features=float(p["max_features"]), random_state=SEED, n_jobs=-1)

def build_knn(p):
    return make_pipeline(StandardScaler(),
                         KNeighborsRegressor(n_neighbors=I(p["n_neighbors"]),
                                             weights="distance" if float(p["distance_weight"]) > 0.5 else "uniform",
                                             p=1 if float(p["p"]) < 1.5 else 2))

def build_svr(p):
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(),
                                SVR(C=10 ** float(p["log10_C"]), gamma=10 ** float(p["log10_gamma"]),
                                    epsilon=float(p["epsilon"]))),
        transformer=StandardScaler())

space = {
    "CatBoost": (build_cat, {"iterations": (200, 800), "depth": (3, 8), "learning_rate": (0.01, 0.3),
                             "l2_leaf_reg": (1, 10), "random_strength": (0.1, 5)}),
    "XGBoost":  (build_xgb, {"n_estimators": (100, 600), "max_depth": (2, 8), "learning_rate": (0.01, 0.3),
                             "subsample": (0.6, 1.0), "colsample_bytree": (0.6, 1.0), "reg_lambda": (0, 10)}),
    "RF":       (build_rf,  {"n_estimators": (100, 500), "max_depth": (3, 20), "min_samples_split": (2, 10),
                             "min_samples_leaf": (1, 5), "max_features": (0.3, 1.0)}),
    "KNN":      (build_knn, {"n_neighbors": (1, 20), "distance_weight": (0, 1), "p": (1, 2)}),
    "SVR":      (build_svr, {"log10_C": (-1, 3), "log10_gamma": (-3, 0.5), "epsilon": (0.001, 0.3)}),
}

def cv_score(build, p, X_tr, y_tr):
    try:
        sc = []
        for tr_i, va_i in kf.split(X_tr):
            m = build(p).fit(X_tr.iloc[tr_i], y_tr.iloc[tr_i])
            sc.append(r2_score(y_tr.iloc[va_i], m.predict(X_tr.iloc[va_i])))
        return float(np.mean(sc))
    except Exception:
        return -1.0

def run_bo2(objective, bounds):
    try:
        from bayes_opt import acquisition
        bo = BayesianOptimization(f=objective, pbounds=bounds,
                                  acquisition_function=acquisition.UpperConfidenceBound(kappa=KAPPA),
                                  random_state=SEED, verbose=0)
        bo.maximize(init_points=N_INIT, n_iter=N_ITER)
    except ImportError:
        from bayes_opt import UtilityFunction
        bo = BayesianOptimization(f=objective, pbounds=bounds, random_state=SEED, verbose=0)
        bo.maximize(init_points=N_INIT, n_iter=N_ITER,
                    acquisition_function=UtilityFunction(kind="ucb", kappa=KAPPA))
    return bo

model_names = ["CatBoost", "XGBoost", "RF", "KNN", "SVR"]

def metrics(y, p):
    rmse = np.sqrt(mean_squared_error(y, p))
    return dict(R2=r2_score(y, p), MAE=mean_absolute_error(y, p), CV_RMSE=(rmse / np.mean(y)) * 100)

# ---------------- Plot helpers ----------------
full_names = {"CatBoost": "Categorical Boosting\n(CatBoost)", "XGBoost": "eXtreme Gradient Boosting\n(XGBoost)",
              "RF": "Random Forest\n(RF)", "KNN": "K-Nearest Neighbors\n(KNN)",
              "SVR": "Support Vector Regression\n(SVR)"}
xlabels = {"EUI": r"Actual EUI $\mathrm{(kWh/(m^2\cdot y))}$",
           "IDD": r"Actual IDD value ($^\circ$C)",
           "OCI": r"Actual OCI (kgCO$_2$-eq/(m$^2\cdot$y))"}
ylabels = {"EUI": r"Predicted EUI $\mathrm{(kWh/(m^2\cdot y))}$",
           "IDD": r"Predicted IDD ($^\circ$C)",
           "OCI": r"Predicted OCI (kgCO$_2$-eq/(m$^2\cdot$y))"}

def fmt(v):
    return f"{v:.4f}" if v < 0.01 else f"{v:.3f}"

def metrics_box(ax, mtr, mte):
    x0, x1, y0, y1 = 0.58, 0.98, 0.02, 0.50
    ax.add_patch(FancyBboxPatch((x0, y0), x1 - x0, y1 - y0,
                 boxstyle="round,pad=0,rounding_size=0.02", transform=ax.transAxes,
                 fc="white", ec="#333333", lw=1.0, alpha=0.9, zorder=5))
    step = (y1 - y0) / 8.6
    def block(title, m, start):
        lines = [r"$\bf{" + title + "}$", f"$R^2$: {m['R2']:.{R2_DEC}f}",
                 f"MAE: {fmt(m['MAE'])}", f"CV-RMSE: {m['CV_RMSE']:.2f}%"]
        for i, s in enumerate(lines):
            ax.text(x1 - 0.02, y1 - (start + i + 0.6) * step, s, transform=ax.transAxes,
                    ha="right", va="center", fontsize=10.6, zorder=6)
    block("TRAIN", mtr, 0)
    ys = y1 - 4.3 * step
    ax.plot([x0 + 0.02, x1 - 0.02], [ys, ys], transform=ax.transAxes, color="#666666",
            lw=0.8, ls="--", zorder=6)
    block("TEST", mte, 4.6)

def plot_fig4(scenario, preds, metrics_df):
    fig, axes = plt.subplots(3, 5, figsize=(21, 13.5))
    for r, t in enumerate(targets):
        allv = np.concatenate([np.concatenate(preds[(m, t)]) for m in model_names])
        pad = 0.06 * (allv.max() - allv.min())
        lo, hi = allv.min() - pad, allv.max() + pad
        for c, name in enumerate(model_names):
            ax = axes[r, c]
            ytr, ptr, yte, pte = preds[(name, t)]
            ax.scatter(ytr, ptr, s=20, c=TRAIN_C, alpha=0.35, linewidths=0, zorder=2)
            ax.scatter(yte, pte, s=22, c=TEST_C, alpha=0.75, linewidths=0, zorder=3)
            ax.plot([lo, hi], [lo, hi], color="#2c3e50", lw=1.8, ls="--", zorder=4)
            ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
            ax.grid(True, linestyle=":", alpha=0.4, zorder=1)
            ax.tick_params(axis='both', which='major', labelsize=12.5)
            ax.set_xlabel(xlabels[t], fontsize=13.8, fontweight="bold")
            if c == 0:
                ax.set_ylabel(ylabels[t], fontsize=13.8, fontweight="bold")
                ax.text(-0.30, 1.03, f"({'abc'[r]})", transform=ax.transAxes, fontsize=17.5,
                        fontweight="bold", va="bottom")
            if r == 0:
                ax.set_title(full_names[name], fontsize=16.25, fontweight="bold", pad=10)
            ax.legend(handles=[
                Line2D([], [], marker="o", ls="", color=TRAIN_C, alpha=0.6, markersize=6, label="Train data"),
                Line2D([], [], marker="o", ls="", color=TEST_C, markersize=6, label="Test data"),
                Line2D([], [], color="#2c3e50", ls="--", lw=1.8, label="Ideal fit ($y=x$)")],
                loc="upper left", fontsize=10.6, frameon=True, facecolor="white", framealpha=0.85,
                edgecolor="none", handletextpad=0.2, labelspacing=0.25)
            mt = metrics_df[(metrics_df.Target == t) & (metrics_df.Model == name)]
            metrics_box(ax, mt[mt.Split == "Train"].iloc[0], mt[mt.Split == "Test"].iloc[0])

    fig.suptitle(f"{scenario.replace('_', '-')} scenario", fontsize=20, fontweight="bold", y=1.01)
    plt.tight_layout(h_pad=2.2, w_pad=1.5)
    plt.savefig(f"Figure4_model_comparison_{scenario}.png", dpi=FIG_DPI, bbox_inches="tight")
    plt.savefig(f"Figure4_model_comparison_{scenario}.pdf", bbox_inches="tight")
    plt.savefig(f"Figure4_model_comparison_{scenario}.tif", dpi=FIG_DPI, bbox_inches="tight",
                pil_kwargs={"compression": "tiff_lzw"})
    plt.show()

# ---------------- Run each scenario separately ----------------
all_metrics = {}
for SCENARIO in ["Baseline", "Mid_future"]:
    preds, rows = {}, []
    for t in targets:
        y_tr, y_te = Y[SCENARIO].loc[idx_train, t], Y[SCENARIO].loc[idx_test, t]
        for name in model_names:
            build, bounds = space[name]
            bo = run_bo2(lambda **p: cv_score(build, p, X_train, y_tr), bounds)
            mdl = build(bo.max["params"]).fit(X_train, y_tr)
            p_tr, p_te = np.ravel(mdl.predict(X_train)), np.ravel(mdl.predict(X_test))
            preds[(name, t)] = (y_tr.values, p_tr, y_te.values, p_te)
            for split, yy, pp in [("Train", y_tr.values, p_tr), ("Test", y_te.values, p_te)]:
                rows.append({"Target": t, "Model": name, "Split": split, **metrics(yy, pp)})
            print(f"Done: {SCENARIO} | {t} | {name} | best CV R2 = {bo.max['target']:.4f}")

    metrics_df = pd.DataFrame(rows)
    all_metrics[SCENARIO] = metrics_df
    print(f"\n=== Model comparison ({SCENARIO}) ===")
    display(metrics_df.round(4))
    metrics_df.to_csv(f"model_comparison_{SCENARIO}.csv", index=False)

    test_df = metrics_df[metrics_df.Split == "Test"]
    print(f"\nBest model per target ({SCENARIO}, test R2):")
    print(test_df.loc[test_df.groupby("Target")["R2"].idxmax(), ["Target", "Model", "R2", "CV_RMSE"]]
          .round(5).to_string(index=False))
    print(f"\nMean test R2 over the 3 targets ({SCENARIO}):")
    print(test_df.groupby("Model")["R2"].mean().sort_values(ascending=False).round(5))

    plot_fig4(SCENARIO, preds, metrics_df)

tableA1_models = pd.concat({k: pd.DataFrame(v[1], index=["Lower", "Upper"]).T for k, v in space.items()})
display(tableA1_models)
tableA1_models.to_csv("Table_A1_all_models.csv")

models, best_params = {}, {}
build, bounds = space["CatBoost"]

for scen_key in ["Baseline", "Mid_future"]:
    for t in targets:
        y_tr = Y[scen_key].loc[idx_train, t]
        bo = run_bo2(lambda **p: cv_score(build, p, X_train, y_tr), bounds)
        best_params[(scen_key, t)] = bo.max["params"]
        models[(scen_key, t)] = build(bo.max["params"]).fit(X_train, y_tr)
        print(f"Done: {scen_key} | {t} | CV R2 = {bo.max['target']:.4f}")

display(pd.DataFrame(best_params).T.round(4))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

predictions_test, results = {}, []
for (scen, t), model in models.items():
    y_te = Y[scen].loc[idx_test, t]
    predictions_test[(scen, t)] = (y_te.values, model.predict(X_test))
    for split, Xs, ys in [("Train", X_train, Y[scen].loc[idx_train, t]), ("Test", X_test, y_te)]:
        p = model.predict(Xs); rmse = np.sqrt(mean_squared_error(ys, p))
        results.append({"Scenario": scen, "Target": t, "Split": split, "R2": r2_score(ys, p),
                        "RMSE": rmse, "MAE": mean_absolute_error(ys, p), "CV_RMSE_%": (rmse / ys.mean()) * 100})
results_df = pd.DataFrame(results)
display(results_df.round(4))
results_df.to_csv("final_model_performance.csv", index=False)

plt.rcParams['font.family'] = 'DejaVu Sans'; plt.rcParams['font.size'] = 11
scenarios = [("Baseline", "a)"), ("Mid_future", "b)")]
targets_info = [("EUI", r"EUI value (kWh/(m$^2$$\cdot$y))", "Prediction results of EUI"),
                ("IDD", r"IDD value ($^\circ$C)", "Prediction results of IDD"),
                ("OCI", r"OCI value (kgCO$_2$-eq/(m$^2$$\cdot$y))", "Prediction results of OCI")]

fig, axes = plt.subplots(2, 3, figsize=(16, 7.5), sharex=True)
for r, (scen_key, _) in enumerate(scenarios):
    for c, (t_code, y_label, title) in enumerate(targets_info):
        ax = axes[r, c]
        y_true, y_pred = predictions_test[(scen_key, t_code)]
        rmse_val = np.sqrt(mean_squared_error(y_true, y_pred))
        ax.plot(y_true, color='#2b5c8f', label='Actual', lw=1.2)
        ax.plot(y_pred, color='#d97724', ls='--', label='Predicted', lw=1.2, alpha=0.85)
        if r == 0: ax.set_title(title, fontweight='bold', fontsize=13, pad=10)
        ax.set_ylabel(y_label, fontweight='bold')
        if r == 1: ax.set_xlabel("Prediction samples", fontweight='bold')
        ax.text(0.95, 0.05, f"RMSE = {rmse_val:.4f}", transform=ax.transAxes, va='bottom', ha='right',
                bbox=dict(boxstyle='square,pad=0.3', fc='white', ec='black', lw=0.8))
        ax.legend(loc='upper left', frameon=True, framealpha=0.9, edgecolor='none')
        ax.grid(True, ls=':', alpha=0.6); ax.tick_params(direction='in', top=True, right=True)
fig.text(0.01, 0.93, 'a)', fontsize=14, fontweight='bold')
fig.text(0.01, 0.48, 'b)', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0.02, 0, 1, 1])

# ذخیره با بالاترین کیفیت (DPI 600) و اضافه شدن نسخه TIFF
plt.savefig("Figure_5_Actual_vs_Predicted.png", dpi=600, bbox_inches='tight')
plt.savefig("Figure_5_Actual_vs_Predicted.pdf", bbox_inches='tight')
plt.savefig("Figure_5_Actual_vs_Predicted.tif", dpi=600, bbox_inches='tight', pil_kwargs={"compression": "tiff_lzw"})

plt.show()

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 13

scenarios = [("Baseline", "a)"), ("Mid_future", "b)")]
targets_info = [
    ("EUI", r"Actual EUI (kWh/(m$^2$$\cdot$y))", r"Predicted EUI (kWh/(m$^2$$\cdot$y))", "Prediction results of EUI"),
    ("IDD", r"Actual IDD ($^\circ$C)", r"Predicted IDD ($^\circ$C)", "Prediction results of IDD"),
    ("OCI", r"Actual OCI (kgCO$_2$-eq/(m$^2$$\cdot$y))", r"Predicted OCI (kgCO$_2$-eq/(m$^2$$\cdot$y))", "Prediction results of OCI")
]

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 12))

for row_idx, (scen_key, label_prefix) in enumerate(scenarios):
    Ydf = Y[scen_key]
    y_tr_scen = Ydf.loc[idx_train]
    y_te_scen = Ydf.loc[idx_test]

    for col_idx, (t_code, x_label, y_label, title) in enumerate(targets_info):
        ax = axes[row_idx, col_idx]
        model = models[(scen_key, t_code)]

        y_tr_true = y_tr_scen[t_code].values
        y_tr_pred = model.predict(X_train)
        y_te_true = y_te_scen[t_code].values
        y_te_pred = model.predict(X_test)

        r2_tr = r2_score(y_tr_true, y_tr_pred)
        mae_tr = mean_absolute_error(y_tr_true, y_tr_pred)
        cv_rmse_tr = (np.sqrt(mean_squared_error(y_tr_true, y_tr_pred)) / y_tr_true.mean()) * 100

        r2_te = r2_score(y_te_true, y_te_pred)
        mae_te = mean_absolute_error(y_te_true, y_te_pred)
        cv_rmse_te = (np.sqrt(mean_squared_error(y_te_true, y_te_pred)) / y_te_true.mean()) * 100

        ax.scatter(y_tr_true, y_tr_pred, color='#d8b2d8', alpha=0.5, s=25, label='Train data', edgecolors='none')
        ax.scatter(y_te_true, y_te_pred, color='#58ca93', alpha=0.7, s=25, label='Test data', edgecolors='none')

        all_vals = np.concatenate([y_tr_true, y_te_true, y_tr_pred, y_te_pred])
        min_val, max_val = all_vals.min(), all_vals.max()
        padding = (max_val - min_val) * 0.05
        lims = [min_val - padding, max_val + padding]

        ax.plot(lims, lims, color='black', linestyle=':', linewidth=1.8, label='Ideal fit (y=x)')
        ax.set_xlim(lims)
        ax.set_ylim(lims)

        if row_idx == 0:
            ax.set_title(title, fontweight='bold', fontsize=17, pad=10)
        ax.set_xlabel(x_label, fontweight='bold', fontsize=15)
        ax.set_ylabel(y_label, fontweight='bold', fontsize=15)

        metrics_text = (
            "TRAIN\n"
            f"R$^2$ : {r2_tr:.3f}\n"
            f"MAE: {mae_tr:.3f}\n"
            f"CV-RMSE: {cv_rmse_tr:.1f}%\n"
            "---------------------\n"
            "TEST\n"
            f"R$^2$ : {r2_te:.3f}\n"
            f"MAE: {mae_te:.3f}\n"
            f"CV-RMSE: {cv_rmse_te:.1f}%"
        )
        ax.text(0.95, 0.05, metrics_text, transform=ax.transAxes, fontsize=14,
                verticalalignment='bottom', horizontalalignment='right', multialignment='center',
                bbox=dict(boxstyle='square,pad=0.5', facecolor='white', edgecolor='black', linewidth=0.8))

        ax.legend(loc='upper left', frameon=True, facecolor='white', framealpha=0.9, edgecolor='none', fontsize=14)
        ax.grid(True, linestyle=':', alpha=0.5, color='gray')
        ax.tick_params(direction='in', top=True, right=True)

fig.text(0.01, 0.94, 'a)', fontsize=15, fontweight='bold')
fig.text(0.01, 0.49, 'b)', fontsize=15, fontweight='bold')

plt.tight_layout(rect=[0.02, 0, 1, 1])

# ذخیره تصویر با بالاترین کیفیت (DPI 600) و اضافه شدن نسخه TIFF
plt.savefig("Figure_6_Performance_Metrics_Comparison.png", dpi=600, bbox_inches='tight')
plt.savefig("Figure_6_Performance_Metrics_Comparison.pdf", bbox_inches='tight')
plt.savefig("Figure_6_Performance_Metrics_Comparison.tif", dpi=600, bbox_inches='tight', pil_kwargs={"compression": "tiff_lzw"})

plt.show()

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import numpy as np
import shap

mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Liberation Serif", "DejaVu Serif"]
mpl.rcParams["axes.unicode_minus"] = False

CMAP = "coolwarm"
BAR_COLOR = "#b9cde5"
BAR_ACCENT_COLOR = "#4a7bb0"
SHOW_BAR_VALUES = False

X_plot = X.copy()
X_norm = (X_plot - X_plot.min()) / (X_plot.max() - X_plot.min())
nice = {"Sensible_HR_Eff": "Sensible HR_Eff"}

def jitter(vals, row_height=0.9, nbins=100):
    q = np.round(nbins * (vals - vals.min()) / (vals.max() - vals.min() + 1e-12))
    ys = np.zeros(len(vals))
    layer, last = 0, -1
    for i in np.argsort(vals):
        if q[i] != last:
            layer = 0
            last = q[i]
        ys[i] = np.ceil(layer / 2) * ((layer % 2) * 2 - 1)
        layer += 1
    return ys * (row_height / (np.max(np.abs(ys)) + 1)) * 0.5 if ys.any() else ys

def beeswarm(ax, sv, Xn, cols):
    imp = np.abs(sv).mean(axis=0)
    order = np.argsort(imp)
    for rank, f in enumerate(order):
        ax.scatter(sv[:, f], rank + jitter(sv[:, f]), c=Xn.iloc[:, f], cmap=CMAP,
                   vmin=0, vmax=1, s=4, alpha=0.9, linewidths=0, rasterized=True)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([nice.get(cols[f], cols[f]) for f in order], fontsize=10)
    ax.axvline(0, color="grey", lw=0.6, alpha=0.6, zorder=0)
    ax.set_ylim(-0.7, len(order) - 0.3)
    ax.set_xlabel("SHAP value (Impact on prediction)", fontsize=10, fontweight="bold")
    ax.tick_params(axis="x", labelsize=9)
    ax.tick_params(axis="y", length=0)
    for s in ["top", "right", "left"]:
        ax.spines[s].set_visible(False)
    return imp, order

def add_importance_bars(ax, imp, order):
    ax2 = ax.twiny()
    ax2.barh(range(len(order)), imp[order], height=0.7, color=BAR_COLOR, zorder=1)
    ax2.set_xlim(0, imp.max() * 1.12)
    ax2.set_ylim(ax.get_ylim())
    ax2.xaxis.set_ticks_position("top")
    ax2.xaxis.set_label_position("top")
    ax2.set_xlabel("Mean |SHAP| (Feature Importance)", fontsize=10, fontweight="bold",
                   labelpad=6, color=BAR_ACCENT_COLOR)
    ax2.tick_params(axis="x", labelsize=9, length=3, colors=BAR_ACCENT_COLOR)
    ax2.tick_params(axis="y", left=False, labelleft=False)
    for s in ["left", "right", "bottom"]:
        ax2.spines[s].set_visible(False)
    ax2.spines["top"].set_color(BAR_ACCENT_COLOR)
    if SHOW_BAR_VALUES:
        for rank, f in enumerate(order):
            ax2.text(imp[f], rank, f" {imp[f]:.2f}", va="center", ha="left",
                     fontsize=8, color=BAR_ACCENT_COLOR)
    ax2.set_zorder(1)
    ax.set_zorder(2)
    ax.patch.set_visible(False)

scenarios = ["Baseline", "Mid_future"]
col_titles = ["Baseline scenario", "Mid-future scenario"]

fig, axes = plt.subplots(3, 2, figsize=(11.5, 13.5))
for r, t in enumerate(targets):
    for c, s in enumerate(scenarios):
        ax = axes[r, c]
        sv = shap.TreeExplainer(models[(s, t)]).shap_values(X_plot)
        imp, order = beeswarm(ax, sv, X_norm, list(X_plot.columns))

        cb = fig.colorbar(ScalarMappable(Normalize(0, 1), CMAP), ax=ax, fraction=0.035, pad=0.02)
        cb.set_label("Feature value", fontsize=10, fontweight="bold")
        cb.ax.tick_params(labelsize=8)
        cb.outline.set_visible(False)

        add_importance_bars(ax, imp, order)
        if r == 0:
            ax.set_title(col_titles[c], fontsize=13, fontweight="bold", pad=38)

    axes[r, 0].text(-0.42, 1.02, f"{'abc'[r]})", transform=axes[r, 0].transAxes,
                    fontsize=13, fontweight="bold", va="bottom")

plt.tight_layout(h_pad=3, w_pad=2)

# ذخیره نمودار با بالاترین کیفیت (DPI 600) و اضافه شدن فرمت TIFF
plt.savefig("SHAP_summary_figure.png", dpi=600, bbox_inches="tight")
plt.savefig("SHAP_summary_figure.pdf", bbox_inches="tight")
plt.savefig("SHAP_summary_figure.tif", dpi=600, bbox_inches="tight", pil_kwargs={"compression": "tiff_lzw"})

plt.show()

import shap
import matplotlib.pyplot as plt
import numpy as np

# تنظیمات عمومی فونت و استایل
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 12

dependence_configs = [
    {"target": "EUI", "feature_x": "Cooling_SP", "label_x": "Cooling setpoint (°C)",
     "label_y": "SHAP value for EUI", "interaction_feature": "Infiltration_rate",
     "label_color": r"Infiltration rate (h$^{-1}$)"},
    {"target": "IDD", "feature_x": "Cooling_SP", "label_x": "Cooling setpoint (°C)",
     "label_y": "SHAP value for IDD", "interaction_feature": "Wall_U",
     "label_color": r"Wall U-value (W/m$^2$K)"},
    {"target": "OCI", "feature_x": "Cooling_SP", "label_x": "Cooling setpoint (°C)",
     "label_y": "SHAP value for OCI", "interaction_feature": "Roof_U",
     "label_color": r"Roof U-value (W/m$^2$K)"},
]

scen_titles = {"Baseline": "Baseline scenario", "Mid_future": "Mid-future scenario"}

for scen_key in ["Baseline", "Mid_future"]:
    fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(19, 5.5), dpi=300)

    for ax, config in zip(axes, dependence_configs):
        t_code = config["target"]
        model = models[(scen_key, t_code)]
        shap_values = shap.TreeExplainer(model)(X)

        feat_x, feat_color = config["feature_x"], config["interaction_feature"]
        feat_idx = list(X.columns).index(feat_x)
        y_shap = shap_values.values[:, feat_idx]

        # افزودن خط مرجع صفر برای تحلیل بهتر SHAP
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.7, zorder=1)

        # رسم نقاط با لبه بسیار ظریف جهت تفکیک بهتر در کیفیت بالا
        sc = ax.scatter(X[feat_x].values, y_shap, c=X[feat_color].values,
                        cmap='plasma', s=25, alpha=0.85, edgecolors='k', linewidths=0.2, zorder=2)

        ax.set_title(t_code, fontweight='bold', fontsize=15, pad=12)
        ax.set_xlabel(config["label_x"], fontweight='bold', fontsize=13)
        ax.set_ylabel(config["label_y"], fontweight='bold', fontsize=13)

        # تنظیمات دقیق‌تر تیک‌ها و قاب محورها
        ax.tick_params(axis='both', which='major', labelsize=11, direction='in', top=True, right=True, length=5, width=0.8)
        for spine in ax.spines.values():
            spine.set_linewidth(0.8)

        # نوار رنگی (Colorbar)
        cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(config["label_color"], fontweight='bold', fontsize=12)
        cbar.ax.tick_params(labelsize=10, direction='in')
        cbar.outline.set_linewidth(0.8)

        ax.grid(False)

    fig.suptitle(scen_titles[scen_key], fontsize=16, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.94])

    # ذخیره خروجی PNG با کیفیت بسیار بالا (600 DPI)
    plt.savefig(f"Figure_8_SHAP_Dependence_{scen_key}.png",
                dpi=600, bbox_inches='tight', facecolor='white', transparent=False)

    # ذخیره خروجی TIF با کیفیت بسیار بالا (600 DPI) و فشرده‌سازی LZW
    plt.savefig(f"Figure_8_SHAP_Dependence_{scen_key}.tif",
                dpi=600, bbox_inches='tight', facecolor='white', transparent=False,
                pil_kwargs={"compression": "tiff_lzw"})

    plt.show()

In [ ]:
# ============ CELL 8: Figure 9 - Distributions of ALL Initial Dataset Solutions ============
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------------------------------------------------
# ۱. ساخت دیتافریم کامل از کل داده‌های اولیه (All Dataset Samples)
# -------------------------------------------------------------------------
df_base_all = pd.concat([X, y_base], axis=1)
df_base_all['Scenario_Label'] = "Baseline\nscenario"

df_future_all = pd.concat([X, y_future], axis=1)
df_future_all['Scenario_Label'] = "Mid-future\nscenario"

# ترکیب تمامی داده‌های اولیه هر دو سناریو
df_all_dataset = pd.concat([df_base_all, df_future_all], ignore_index=True)

print(f"تعداد کل داده‌های اولیه بررسی‌شده در نمودار: {len(df_all_dataset)} نمونه")

# -------------------------------------------------------------------------
# ۲. تنظیمات عمومی و استایل نمودار Figure 9
# -------------------------------------------------------------------------
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['axes.linewidth'] = 0.8

fig, axes = plt.subplots(1, 3, figsize=(15, 5.5), sharex=True, dpi=300)

target_configs = [
    {
        'col': 'EUI',
        'title': 'EUI distribution',
        'ylabel': r'EUI value (kWh/(m$^2$$\cdot$y))',
        'ax': axes[0]
    },
    {
        'col': 'IDD',
        'title': 'IDD distribution',
        'ylabel': r'IDD value ($\degree$C)',
        'ax': axes[1]
    },
    {
        'col': 'OCI',
        'title': 'OCI distribution',
        'ylabel': r'OCI value (kgCO$_2$-eq/(m$^2$$\cdot$y))',
        'ax': axes[2]
    }
]

colors = ['#2b93b8', '#e06138']
scenarios_labels = ["Baseline\nscenario", "Mid-future\nscenario"]

# -------------------------------------------------------------------------
# ۳. رسم subplot‌ها (Violin Plot + BoxPlot) بر روی کل داده‌ها
# -------------------------------------------------------------------------
for config in target_configs:
    ax = config['ax']
    col = config['col']

    # ۱. رسم نمودار ویولن برای کل داده‌های اولیه
    sns.violinplot(
        data=df_all_dataset,
        x='Scenario_Label',
        y=col,
        hue='Scenario_Label',
        legend=False,
        ax=ax,
        palette=colors,
        inner=None,        # عدم رسم خطوط داخلی پیش‌فرض جهت رسم باکس‌پلات سفارشی
        cut=0,             # محدود کردن به دامنه داده‌های واقعی
        linewidth=0.9,
        saturation=0.95
    )

    # ۲. افزودن باکس‌پلات دقیق در مرکز هر ویولن
    for i, scen_label in enumerate(scenarios_labels):
        vals = df_all_dataset[df_all_dataset['Scenario_Label'] == scen_label][col].dropna().values

        q25, median_val, q75 = np.percentile(vals, [25, 50, 75])
        iqr = q75 - q25

        # اصلاح علامت ضرب در محاسبه ویسکرها
        whisker_low_candidates = vals[vals >= q25 - 1.5 * iqr]
        whisker_high_candidates = vals[vals <= q75 + 1.5 * iqr]

        whisker_low = np.min(whisker_low_candidates) if len(whisker_low_candidates) > 0 else q25
        whisker_high = np.max(whisker_high_candidates) if len(whisker_high_candidates) > 0 else q75

        # رسم خطوط Whisker (داده‌های حدی)
        ax.plot([i, i], [whisker_low, whisker_high], color='#2c3e50', linewidth=1.2, zorder=3)
        ax.plot([i - 0.04, i + 0.04], [whisker_low, whisker_low], color='#2c3e50', linewidth=1.2, zorder=3)
        ax.plot([i - 0.04, i + 0.04], [whisker_high, whisker_high], color='#2c3e50', linewidth=1.2, zorder=3)

        # رسم جعبه بین چارک ۲۵ و ۷۵
        rect = plt.Rectangle(
            (i - 0.065, q25), 0.13, q75 - q25,
            facecolor='#34495e', edgecolor='#2c3e50', linewidth=0.8, alpha=0.9, zorder=4
        )
        ax.add_patch(rect)

        # خط افقی میانه به رنگ سفید
        ax.plot([i - 0.065, i + 0.065], [median_val, median_val], color='white', linewidth=1.2, zorder=5)

    # تنظیمات ظاهری عنوان و محورها
    ax.set_title(config['title'], fontweight='bold', fontsize=14, pad=12)
    ax.set_ylabel(config['ylabel'], fontweight='bold', fontsize=12)
    ax.set_xlabel('', labelpad=0)

    ax.tick_params(axis='x', labelsize=11.5)
    ax.tick_params(axis='y', labelsize=11, direction='in', top=True, right=True)

    # شبکه‌بندی پس‌زمینه
    ax.yaxis.grid(True, linestyle=':', alpha=0.5, color='gray')
    ax.set_axisbelow(True)

plt.tight_layout()

# ذخیره خروجی PNG با کیفیت بالا (600 DPI)
plt.savefig("Figure_9_Violin_Distributions_All_Data.png",
            dpi=600, bbox_inches='tight', facecolor='white', transparent=False)

# ذخیره خروجی TIF با کیفیت بالا (600 DPI) و فشرده‌سازی LZW
plt.savefig("Figure_9_Violin_Distributions_All_Data.tif",
            dpi=600, bbox_inches='tight', facecolor='white', transparent=False,
            pil_kwargs={"compression": "tiff_lzw"})

# ذخیره خروجی PDF (برداری)
plt.savefig("Figure_9_Violin_Distributions_All_Data.pdf", bbox_inches='tight')

plt.show()

In [ ]:
# ============ CELL 9: NSGA-II Optimization & TOPSIS Ranking (Top 50 Pareto, corrected) ============
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize

In [ ]:
# ---------------- 1. Settings ----------------
DISCRETE = True
POP_SIZE, N_GEN = 200, 200
CROSS_PROB, CROSS_ETA = 0.9, 15
MUT_PROB, MUT_ETA = 1 / len(features), 20
SEED = 42
N_DISPLAY_PARETO = 50

In [ ]:
# ---------------- 2. Fixed variables & bounds ----------------
# Confirm with the SHAP summary that Floor_U is unimportant in BOTH scenarios.
FIXED_VARIABLES = {}
if "Floor_U" in features:
    floor_grid = np.sort(X["Floor_U"].dropna().unique().astype(float))
    FIXED_VARIABLES["Floor_U"] = float(floor_grid[len(floor_grid) // 2])
    print(f"Floor_U fixed at {FIXED_VARIABLES['Floor_U']:.4f}.")

xl = X[features].min().values.astype(float)
xu = X[features].max().values.astype(float)
grids = {c: np.sort(X[c].dropna().unique().astype(float)) for c in features}

def snap(x):
    x = np.asarray(x, dtype=float).copy()
    if x.ndim == 1:
        x = x.reshape(1, -1)
    for j, c in enumerate(features):
        g = grids[c]
        x[:, j] = g[np.abs(x[:, j, None] - g[None, :]).argmin(axis=1)]
    for c, v in FIXED_VARIABLES.items():
        j = features.index(c)
        g = grids[c]
        x[:, j] = g[np.abs(g - v).argmin()]
    return x

In [ ]:
# ---------------- 3. Problem ----------------
class RetrofitProblem(Problem):
    def __init__(self, scenario):
        super().__init__(n_var=len(features), n_obj=3, xl=xl, xu=xu)
        self.m = [models[(scenario, t)] for t in targets]

    def _evaluate(self, x, out, *args, **kwargs):
        xe = snap(x) if DISCRETE else x
        out["F"] = np.column_stack([mdl.predict(xe) for mdl in self.m])

In [ ]:
# ---------------- 4. Optimization + TOPSIS (each scenario separately) ----------------
pareto_dict, best_dict, comparison_rows = {}, {}, []
Yd = {"Baseline": y_base, "Mid_future": y_future}

for scenario in ["Baseline", "Mid_future"]:
    print(f"\n================ Running Optimization for: {scenario} ================")
    algorithm = NSGA2(
        pop_size=POP_SIZE,
        sampling=FloatRandomSampling(),
        crossover=SBX(prob=CROSS_PROB, eta=CROSS_ETA),
        mutation=PM(prob_var=MUT_PROB, eta=MUT_ETA),
        eliminate_duplicates=True,
    )
    res = minimize(RetrofitProblem(scenario), algorithm, ("n_gen", N_GEN), seed=SEED, verbose=False)

    X_par = snap(res.X) if DISCRETE else res.X
    pareto = pd.DataFrame(X_par, columns=features)
    pareto[targets] = res.F
    pareto = pareto.drop_duplicates(subset=targets).reset_index(drop=True)
    pareto["Scenario"] = scenario
    print(f"Pareto-optimal solutions found: {len(pareto)} | Scenario: {scenario}")

    # Entropy weights
    F = pareto[targets].values.astype(float)
    n_sol = F.shape[0]
    if n_sol < 2:
        raise ValueError("At least two Pareto solutions are required.")
    F_min, F_max = F.min(axis=0), F.max(axis=0)
    F_range = F_max - F_min
    F_shift = np.ones_like(F)
    for j in range(F.shape[1]):
        if F_range[j] > 1e-12:
            F_shift[:, j] = (F[:, j] - F_min[j]) / F_range[j]
    F_shift = F_shift + 1e-9
    P = F_shift / F_shift.sum(axis=0)
    entropy = -(P * np.log(P)).sum(axis=0) / np.log(n_sol)
    c_j = 1 - entropy
    w = np.ones(len(targets)) / len(targets) if c_j.sum() <= 1e-12 else c_j / c_j.sum()
    print("Entropy weights:", {t: round(float(v), 4) for t, v in zip(targets, w)})

    # TOPSIS (all objectives are cost-type)
    den = np.sqrt((F ** 2).sum(axis=0))
    den[den < 1e-12] = 1.0
    V = (F / den) * w
    ideal, anti = V.min(axis=0), V.max(axis=0)
    d_plus = np.sqrt(((V - ideal) ** 2).sum(axis=1))
    d_minus = np.sqrt(((V - anti) ** 2).sum(axis=1))
    s = d_plus + d_minus
    pareto["C_star"] = np.where(s > 1e-12, d_minus / s, 0.5)
    pareto = pareto.sort_values("C_star", ascending=False).reset_index(drop=True)
    best = pareto.iloc[0]

    pareto_dict[scenario] = pareto
    best_dict[scenario] = best

    # Optimal vs reference (reference = mean of sampled designs, NOT a real building)
    x_best = best[features].values.astype(float).reshape(1, -1)
    for t in targets:
        ref = Yd[scenario][t].mean()
        opt = float(models[(scenario, t)].predict(x_best)[0])
        comparison_rows.append({
            "Scenario": scenario, "Target": t,
            "Reference (mean of sampled designs)": ref,
            "Optimal design": opt,
            "Change_%": (opt - ref) / ref * 100 if abs(ref) > 1e-12 else np.nan
        })

    pareto.to_csv(f"pareto_solutions_{scenario}.csv", index=False)

all_pareto = pd.concat([pareto_dict["Baseline"], pareto_dict["Mid_future"]], ignore_index=True)
all_pareto.to_csv("pareto_solutions_all_scenarios.csv", index=False)

comparison = pd.DataFrame(comparison_rows)
print("\n=== Optimal design vs reference (both scenarios) ===")
display(comparison.round(3))
print("NOTE: reference = mean of sampled designs, not a measured existing building.")

In [ ]:
# ---------------- 5. Selected designs (needed for the paper) ----------------
selected = pd.DataFrame(best_dict).T[features + targets + ["C_star"]].astype(float)
print("\n=== Selected TOPSIS design per scenario ===")
display(selected.round(4))

comparison.to_csv("optimal_vs_reference.csv", index=False)
selected.to_csv("selected_TOPSIS_designs.csv")

In [ ]:
# ---------------- 6. Plot (top 50 TOPSIS-ranked Pareto solutions) ----------------
plt.rcParams.update({
    "font.sans-serif": ["DejaVu Sans", "Arial", "Liberation Sans", "sans-serif"],
    "font.family": "sans-serif",
    "axes.edgecolor": "#333333", "axes.linewidth": 0.8,
    "grid.color": "#e0e0e0", "grid.linestyle": "--", "grid.linewidth": 0.5
})

target_labels = {
    "EUI": r"EUI (kWh/(m$^2$$\cdot$y))",
    "IDD": r"IDD ($^\circ$C)",
    "OCI": r"OCI (kgCO$_2$-eq/(m$^2$$\cdot$y))"
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
sc_mappable = None

for row_idx, scen in enumerate(["Baseline", "Mid_future"]):
    p_df = pareto_dict[scen].head(N_DISPLAY_PARETO)
    b_df = best_dict[scen]
    for k, (a, b) in enumerate([("EUI", "IDD"), ("EUI", "OCI"), ("IDD", "OCI")]):
        ax = axes[row_idx, k]
        sc = ax.scatter(p_df[a], p_df[b], c=p_df["C_star"], cmap="viridis", s=45,
                        edgecolors="black", linewidths=0.4, alpha=0.9, zorder=3)
        ax.scatter(b_df[a], b_df[b], c="#D62728", s=220, marker="*", edgecolors="black",
                   linewidths=0.6, zorder=5,
                   label="Top TOPSIS design" if (row_idx == 0 and k == 0) else "")
        ax.set_title(f"{scen.replace('_', '-')}: {a} vs {b}", fontsize=12, fontweight="bold", pad=8)
        ax.set_xlabel(target_labels[a], fontsize=10)
        ax.set_ylabel(target_labels[b], fontsize=10)
        ax.grid(True)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
        if row_idx == 0 and k == 0:
            ax.legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.9, fontsize=9)
        sc_mappable = sc

cbar_ax = fig.add_axes([0.25, 0.02, 0.5, 0.02])
cbar = fig.colorbar(sc_mappable, cax=cbar_ax, orientation="horizontal")
cbar.set_label(r"TOPSIS closeness coefficient ($C^*$), top 50 solutions per scenario",
               fontsize=11, fontweight="bold")

plt.subplots_adjust(top=0.93, bottom=0.12, hspace=0.35, wspace=0.28)

plt.savefig("Pareto_fronts_top50.tif", format="tiff", dpi=600, bbox_inches="tight",
            pil_kwargs={"compression": "tiff_lzw"})
plt.savefig("Pareto_fronts_top50.png", format="png", dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved: Pareto_fronts_top50.tif/.png, pareto_solutions_*.csv, "
      "optimal_vs_reference.csv, selected_TOPSIS_designs.csv")

In [ ]:
# ============ CELL 10: Figure 10 - Parallel Coordinates (Top 50) + CSV export ============
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator

N_DISPLAY_PARETO = 50
pareto_base = pareto_dict["Baseline"].head(N_DISPLAY_PARETO).copy()
pareto_mid  = pareto_dict["Mid_future"].head(N_DISPLAY_PARETO).copy()
best_base, best_mid = best_dict["Baseline"].copy(), best_dict["Mid_future"].copy()

# ---------- Initial scheme, evaluated separately in EACH scenario ----------
initial_inputs = {"Cooling_SP": 24.0, "Wall_U": 0.60, "Window_U": 2.40, "Floor_U": 0.40,
                  "Roof_U": 0.35, "Window_SHGC": 0.30, "Infiltration_rate": 0.50, "Sensible_HR_Eff": 0.75}
x_init = pd.DataFrame([initial_inputs])[features]
initial = {}
for scen in ["Baseline", "Mid_future"]:
    d = dict(initial_inputs)
    for t in targets:
        d[t] = float(models[(scen, t)].predict(x_init)[0])
    initial[scen] = d

# ---------- Normalization bounds ----------
all_cols = features + targets
bounds_min = {c: float(X[c].min()) for c in features}
bounds_max = {c: float(X[c].max()) for c in features}
for t in targets:
    v = np.concatenate([pareto_base[t].values, pareto_mid[t].values,
                        [initial["Baseline"][t], initial["Mid_future"][t]]])
    bounds_min[t], bounds_max[t] = float(v.min()), float(v.max())

def normalize_row(r):
    out = []
    for c in all_cols:
        mn, mx = bounds_min[c], bounds_max[c]
        out.append((r[c] - mn) / (mx - mn) if mx > mn else 0.5)
    return np.array(out)

norm_base_p = np.array([normalize_row(r) for _, r in pareto_base.iterrows()])
norm_mid_p  = np.array([normalize_row(r) for _, r in pareto_mid.iterrows()])
norm_best_base, norm_best_mid = normalize_row(best_base), normalize_row(best_mid)
norm_init_base, norm_init_mid = normalize_row(initial["Baseline"]), normalize_row(initial["Mid_future"])

# ---------- CSV export (all inputs and outputs) ----------
def block(df, scen, kind):
    d = df[features + targets + ["C_star"]].astype(float).copy() if "C_star" in df else df.copy()
    d.insert(0, "Rank", range(1, len(d) + 1))
    d.insert(0, "Type", kind)
    d.insert(0, "Scenario", scen)
    return d

best_b = pd.DataFrame([best_base[features + targets + ["C_star"]].astype(float)])
best_m = pd.DataFrame([best_mid[features + targets + ["C_star"]].astype(float)])
init_rows = []
for scen in ["Baseline", "Mid_future"]:
    r = {"Scenario": scen, "Type": "Initial scheme", "Rank": np.nan, **initial[scen], "C_star": np.nan}
    init_rows.append(r)

fig10_data = pd.concat([
    block(pareto_base, "Baseline", "Top-50 Pareto"),
    block(pareto_mid, "Mid_future", "Top-50 Pareto"),
    block(best_b, "Baseline", "Selected TOPSIS design"),
    block(best_m, "Mid_future", "Selected TOPSIS design"),
    pd.DataFrame(init_rows)], ignore_index=True)
fig10_data = fig10_data[["Scenario", "Type", "Rank"] + features + targets + ["C_star"]]
fig10_data.to_csv("Figure_10_data_inputs_outputs.csv", index=False)

pd.DataFrame({"Min": bounds_min, "Max": bounds_max}).loc[all_cols].to_csv("Figure_10_axis_bounds.csv")
print("Rows saved:", len(fig10_data))
display(fig10_data[fig10_data.Type != "Top-50 Pareto"].round(4))

# ---------- Plot ----------
plt.rcParams.update({"font.sans-serif": ["DejaVu Sans", "Arial", "Liberation Sans", "sans-serif"],
                     "font.family": "sans-serif"})
fig, ax = plt.subplots(figsize=(15, 8))
n_vars = len(all_cols)
x_idx = np.arange(n_vars)
x_smooth = np.linspace(0, n_vars - 1, 300)

def smooth(y):
    return np.clip(PchipInterpolator(x_idx, y)(x_smooth), 0.0, 1.0)

ax.fill_between(x_smooth, smooth(norm_base_p.min(0)), smooth(norm_base_p.max(0)),
                color='#b0e0e6', alpha=0.45, label='Top 50 Pareto solutions (Baseline)')
ax.fill_between(x_smooth, smooth(norm_mid_p.min(0)), smooth(norm_mid_p.max(0)),
                color='#ffc0cb', alpha=0.45, label='Top 50 Pareto solutions (Mid-future)')

ax.plot(x_smooth, smooth(norm_best_base), color='#008080', lw=2.5, label='Ideal solution (Baseline)')
ax.plot(x_smooth, smooth(norm_best_mid),  color='#c71585', lw=2.5, label='Ideal solution (Mid-future)')
ax.plot(x_smooth, smooth(norm_init_base), color='black', ls='--', lw=2.0, label='Initial scheme (Baseline)')
ax.plot(x_smooth, smooth(norm_init_mid),  color='dimgray', ls=':', lw=2.2, label='Initial scheme (Mid-future)')

ax.scatter(x_idx, norm_best_base, color='#008080', s=40, zorder=5)
ax.scatter(x_idx, norm_best_mid,  color='#c71585', s=40, zorder=5)
ax.scatter(x_idx, norm_init_base, color='black', s=30, zorder=5)
ax.scatter(x_idx, norm_init_mid,  color='dimgray', s=30, zorder=5)

for i in range(n_vars):
    ax.axvline(i, color='lightgray', lw=0.8, zorder=1)
n_features = len(features)
ax.axvline(n_features - 0.5, color='gray', ls='--', lw=1.5, zorder=2)

labels_map = {"Sensible_HR_Eff": "Sensible HR_Eff",
              "EUI": r"EUI (kWh/(m$^2$$\cdot$y))", "IDD": r"IDD ($^\circ$C)",
              "OCI": r"OCI (kgCO$_2$-eq/(m$^2$$\cdot$y))"}
ax.set_xticks(x_idx)
ax.set_xticklabels([labels_map.get(c, c) for c in all_cols], rotation=25, ha='right',
                   fontweight='bold', fontsize=10)

for i, col in enumerate(all_cols):
    mn, mx = bounds_min[col], bounds_max[col]
    ax.text(i, 1.03, f"{mx:.2f}" if mx < 10 else f"{mx:.1f}", ha='center', va='bottom', fontsize=9.5, fontweight='bold')
    ax.text(i, -0.05, f"{mn:.2f}" if mn < 10 else f"{mn:.1f}", ha='center', va='top', fontsize=9.5, fontweight='bold')

ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0%', '25%', '50%', '75%', '100%'], fontweight='bold')
ax.set_ylabel("Normalization", fontweight='bold', fontsize=12)
ax.set_ylim(-0.08, 1.08)

ax.text((n_features - 1) / 2.0, 1.12, "Retrofit variables", ha='center', va='center', fontsize=12,
        fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', facecolor='#f0f0f0', edgecolor='none'))
ax.text(n_features + (len(targets) - 1) / 2.0, 1.12, "Performance results", ha='center', va='center',
        fontsize=12, fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', facecolor='#f0f0f0', edgecolor='none'))

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=3, frameon=True,
          facecolor='white', edgecolor='gray', fontsize=10.5)
ax.grid(False)
plt.tight_layout()

plt.savefig("Figure_10_Parallel_Coordinates_top50.tif", format="tiff", dpi=600, bbox_inches="tight",
            pil_kwargs={"compression": "tiff_lzw"})
plt.savefig("Figure_10_Parallel_Coordinates_top50.png", format="png", dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved: Figure_10_Parallel_Coordinates_top50.tif/.png, "
      "Figure_10_data_inputs_outputs.csv, Figure_10_axis_bounds.csv")

In [ ]:
# ============ CELL 11: Figure 11 - Spearman heatmap (8 inputs, Top 50 Pareto, clean version) ============
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from scipy.stats import spearmanr

N_DISPLAY_PARETO = 50
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "Liberation Serif"],
    "mathtext.fontset": "stix",
    "axes.unicode_minus": False
})

nice = {
    "Sensible_HR_Eff": "Sensible HR_Eff", "Cooling_SP": "Cooling SP", "Infiltration_rate": "Infiltration",
    "Window_SHGC": "Window SHGC", "Wall_U": "Wall U", "Window_U": "Window U",
    "Floor_U": "Floor U", "Roof_U": "Roof U"
}
cmap = LinearSegmentedColormap.from_list("corr", ["#2f6ea5", "#f7f7f7", "#c0432b"])
norm = Normalize(-1, 1)
cols_all = list(features)
n = len(cols_all)
scens = ["Baseline", "Mid_future"]
titles = {"Baseline": "Baseline scenario", "Mid_future": "Mid-future scenario"}
data = {s: pareto_dict[s].head(N_DISPLAY_PARETO) for s in scens}

def spearman_matrix(df, cols):
    rho = np.full((len(cols), len(cols)), np.nan); pv = rho.copy()
    for i in range(len(cols)):
        for j in range(len(cols)):
            ok = df[cols[i]].nunique() > 1 and df[cols[j]].nunique() > 1
            if i == j and ok:
                rho[i, j], pv[i, j] = 1.0, 0.0
            elif i != j and ok:
                rho[i, j], pv[i, j] = spearmanr(df[cols[i]].values, df[cols[j]].values)
    return pd.DataFrame(rho, index=cols, columns=cols), pd.DataFrame(pv, index=cols, columns=cols)

stars = lambda p: "**" if p <= 0.01 else ("*" if p <= 0.05 else "")
fmt_r = lambda r: f"{r:.2f}" if abs(r) >= 0.01 else f"{r:.3f}"
radius = lambda r: 0.08 + 0.34 * abs(r)
radius_leg = lambda r: 0.05 + 0.18 * abs(r)

fig, axes = plt.subplots(1, 2, figsize=(16, 9.2))
fig.subplots_adjust(left=0.07, right=0.985, top=0.91, bottom=0.30, wspace=0.12)
rho_all, p_all = {}, {}

for ax, scen in zip(axes, scens):
    df = data[scen]
    rho, pval = spearman_matrix(df, cols_all)
    rho_all[scen], p_all[scen] = rho, pval
    R, P = rho.values, pval.values

    for i in range(n):
        for j in range(n):
            if np.isnan(R[i, j]):
                ax.text(j, i, "–", ha="center", va="center", fontsize=13, color="black")
                continue
            if i == j:
                ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, facecolor="#f0f0f0", lw=0, zorder=1))
                ax.text(j, i, "1", ha="center", va="center", fontsize=10, color="black", fontweight="bold")
            elif i > j:
                ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, facecolor=cmap(norm(R[i, j])), lw=0, zorder=1))
                tc = "white" if abs(R[i, j]) > 0.55 else "black"
                ax.text(j, i - 0.07, fmt_r(R[i, j]), ha="center", va="center", fontsize=10.5,
                        fontweight="bold", color=tc, zorder=3)
                s = stars(P[i, j])
                if s:
                    ax.text(j, i + 0.24, s, ha="center", va="center", fontsize=10, fontweight="bold",
                            color=tc, zorder=3)
            else:
                ax.add_patch(plt.Circle((j, i), radius(R[i, j]), facecolor=cmap(norm(R[i, j])),
                                        edgecolor="black", lw=0.8, zorder=3))

    for k in np.arange(-.5, n, 1):
        ax.axhline(k, color="#cccccc", lw=0.8, zorder=2); ax.axvline(k, color="#cccccc", lw=0.8, zorder=2)

    ax.set_xlim(-.5, n - .5); ax.set_ylim(n - .5, -.5); ax.set_aspect("equal")
    labels = [nice[c] for c in cols_all]
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=40, ha="right", rotation_mode="anchor", fontsize=10.5, fontweight="bold")
    ax.set_yticklabels(labels, fontsize=10.5, fontweight="bold")
    ax.tick_params(length=0, pad=4)
    for sp in ax.spines.values():
        sp.set_linewidth(1.2); sp.set_color("black")
    ax.set_title(f"{titles[scen]} (top {N_DISPLAY_PARETO} Pareto solutions)", fontsize=13.5,
                 fontweight="bold", pad=8)

# ---- Legends: Clean layout with absolute separation ----
LEG_Y = 0.08
cax = fig.add_axes([0.07, LEG_Y + 0.02, 0.22, 0.02])
cb = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), cax=cax, orientation="horizontal",
                  ticks=[-1, -0.5, 0, 0.5, 1])
cb.ax.set_xticklabels(["−1", "−0.5", "0", "0.5", "1"], fontsize=10)
cb.outline.set_edgecolor("black")
cb.set_label("Spearman ρ", fontsize=11, fontweight="bold")

# Circle size legend with fixed aspect ratio to prevent oval distortion
fig.text(0.43, LEG_Y + 0.075, "Circle size (upper triangle)", fontsize=10, fontweight="bold", ha="center")
lax = fig.add_axes([0.33, LEG_Y - 0.02, 0.20, 0.08])
lax.axis("off")
lax.set_xlim(0, 3)
lax.set_ylim(0, 1)
lax.set_aspect("equal") # تنظیم نسبت ابعاد برای جلوگیری از بیضی شدن دایره‌ها

for k, v in enumerate([0.1, 0.5, 0.9]):
    lax.add_patch(plt.Circle((0.5 + k * 1.0, 0.5), radius_leg(v) * 1.5, facecolor="#bdbdbd",
                             edgecolor="black", lw=0.8))
    lax.text(0.5 + k * 1.0, -0.25, f"|ρ| = {v}", ha="center", va="center", fontsize=9.5)

fig.text(0.64, LEG_Y + 0.055, "* p ≤ 0.05     ** p ≤ 0.01", fontsize=10, style="italic")
fig.text(0.64, LEG_Y + 0.015, "–  correlation undefined (constant variable)", fontsize=10, style="italic")

plt.savefig("Figure11_Spearman_heatmap_top50.tif", format="tiff", dpi=600, bbox_inches="tight",
            pil_kwargs={"compression": "tiff_lzw"})
plt.savefig("Figure11_Spearman_heatmap_top50.png", dpi=300, bbox_inches="tight")
plt.show()

for s in scens:
    rho_all[s].round(4).to_csv(f"spearman_rho_{s}_top50.csv")
    p_all[s].to_csv(f"spearman_pvalues_{s}_top50.csv")
print("Saved: Figure11_Spearman_heatmap_top50.tif/.png and CSV files")

In [ ]:
# ============ CELL 11: Figure 11 - Pearson heatmap (8 inputs, Top 50 Pareto, clean version) ============
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from scipy.stats import pearsonr

N_DISPLAY_PARETO = 50
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "Liberation Serif"],
    "mathtext.fontset": "stix",
    "axes.unicode_minus": False
})

nice = {
    "Sensible_HR_Eff": "Sensible HR_Eff", "Cooling_SP": "Cooling SP", "Infiltration_rate": "Infiltration",
    "Window_SHGC": "Window SHGC", "Wall_U": "Wall U", "Window_U": "Window U",
    "Floor_U": "Floor U", "Roof_U": "Roof U"
}
cmap = LinearSegmentedColormap.from_list("corr", ["#2f6ea5", "#f7f7f7", "#c0432b"])
norm = Normalize(-1, 1)
cols_all = list(features)
n = len(cols_all)
scens = ["Baseline", "Mid_future"]
titles = {"Baseline": "Baseline scenario", "Mid_future": "Mid-future scenario"}
data = {s: pareto_dict[s].head(N_DISPLAY_PARETO) for s in scens}

def pearson_matrix(df, cols):
    r_mat = np.full((len(cols), len(cols)), np.nan); pv = r_mat.copy()
    for i in range(len(cols)):
        for j in range(len(cols)):
            ok = df[cols[i]].nunique() > 1 and df[cols[j]].nunique() > 1
            if i == j and ok:
                r_mat[i, j], pv[i, j] = 1.0, 0.0
            elif i != j and ok:
                r_mat[i, j], pv[i, j] = pearsonr(df[cols[i]].values, df[cols[j]].values)
    return pd.DataFrame(r_mat, index=cols, columns=cols), pd.DataFrame(pv, index=cols, columns=cols)

stars = lambda p: "**" if p <= 0.01 else ("*" if p <= 0.05 else "")
fmt_r = lambda r: f"{r:.2f}" if abs(r) >= 0.01 else f"{r:.3f}"
radius = lambda r: 0.08 + 0.34 * abs(r)
radius_leg = lambda r: 0.05 + 0.18 * abs(r)

fig, axes = plt.subplots(1, 2, figsize=(16, 9.2))
fig.subplots_adjust(left=0.07, right=0.985, top=0.91, bottom=0.30, wspace=0.12)
r_all, p_all = {}, {}

for ax, scen in zip(axes, scens):
    df = data[scen]
    r_mat, pval = pearson_matrix(df, cols_all)
    r_all[scen], p_all[scen] = r_mat, pval
    R, P = r_mat.values, pval.values

    for i in range(n):
        for j in range(n):
            if np.isnan(R[i, j]):
                ax.text(j, i, "–", ha="center", va="center", fontsize=13, color="black")
                continue
            if i == j:
                ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, facecolor="#f0f0f0", lw=0, zorder=1))
                ax.text(j, i, "1", ha="center", va="center", fontsize=10, color="black", fontweight="bold")
            elif i > j:
                ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, facecolor=cmap(norm(R[i, j])), lw=0, zorder=1))
                tc = "white" if abs(R[i, j]) > 0.55 else "black"
                ax.text(j, i - 0.07, fmt_r(R[i, j]), ha="center", va="center", fontsize=10.5,
                        fontweight="bold", color=tc, zorder=3)
                s = stars(P[i, j])
                if s:
                    ax.text(j, i + 0.24, s, ha="center", va="center", fontsize=10, fontweight="bold",
                            color=tc, zorder=3)
            else:
                ax.add_patch(plt.Circle((j, i), radius(R[i, j]), facecolor=cmap(norm(R[i, j])),
                                        edgecolor="black", lw=0.8, zorder=3))

    for k in np.arange(-.5, n, 1):
        ax.axhline(k, color="#cccccc", lw=0.8, zorder=2); ax.axvline(k, color="#cccccc", lw=0.8, zorder=2)

    ax.set_xlim(-.5, n - .5); ax.set_ylim(n - .5, -.5); ax.set_aspect("equal")
    labels = [nice[c] for c in cols_all]
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=40, ha="right", rotation_mode="anchor", fontsize=10.5, fontweight="bold")
    ax.set_yticklabels(labels, fontsize=10.5, fontweight="bold")
    ax.tick_params(length=0, pad=4)
    for sp in ax.spines.values():
        sp.set_linewidth(1.2); sp.set_color("black")
    ax.set_title(f"{titles[scen]} (top {N_DISPLAY_PARETO} Pareto solutions)", fontsize=13.5,
                 fontweight="bold", pad=8)

# ---- Legends: Clean layout with absolute separation ----
LEG_Y = 0.08
cax = fig.add_axes([0.07, LEG_Y + 0.02, 0.22, 0.02])
cb = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), cax=cax, orientation="horizontal",
                  ticks=[-1, -0.5, 0, 0.5, 1])
cb.ax.set_xticklabels(["−1", "−0.5", "0", "0.5", "1"], fontsize=10)
cb.outline.set_edgecolor("black")
cb.set_label("Pearson r", fontsize=11, fontweight="bold")

# Circle size legend with equal aspect ratio to keep circles perfectly round
fig.text(0.43, LEG_Y + 0.075, "Circle size (upper triangle)", fontsize=10, fontweight="bold", ha="center")
lax = fig.add_axes([0.33, LEG_Y - 0.02, 0.20, 0.08])
lax.axis("off")
lax.set_xlim(0, 3)
lax.set_ylim(0, 1)
lax.set_aspect("equal")

for k, v in enumerate([0.1, 0.5, 0.9]):
    lax.add_patch(plt.Circle((0.5 + k * 1.0, 0.5), radius_leg(v) * 1.5, facecolor="#bdbdbd",
                             edgecolor="black", lw=0.8))
    lax.text(0.5 + k * 1.0, -0.25, f"|r| = {v}", ha="center", va="center", fontsize=9.5)

fig.text(0.64, LEG_Y + 0.055, "* p ≤ 0.05     ** p ≤ 0.01", fontsize=10, style="italic")
fig.text(0.64, LEG_Y + 0.015, "–  correlation undefined (constant variable)", fontsize=10, style="italic")

plt.savefig("Figure11_Pearson_heatmap_top50.tif", format="tiff", dpi=600, bbox_inches="tight",
            pil_kwargs={"compression": "tiff_lzw"})
plt.savefig("Figure11_Pearson_heatmap_top50.png", dpi=300, bbox_inches="tight")
plt.show()

for s in scens:
    r_all[s].round(4).to_csv(f"pearson_r_{s}_top50.csv")
    p_all[s].to_csv(f"pearson_pvalues_{s}_top50.csv")
print("Saved: Figure11_Pearson_heatmap_top50.tif/.png and CSV files")

In [ ]:
# ============ CELL 8: Table 3 - Pairwise Relationships Among Pareto Objectives (Top 50 Pareto) ============
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr, linregress

In [ ]:
# ---------------- 1. تنظیمات و ۵۰ راهکار برتر پارتو ----------------
N_DISPLAY_PARETO = 50

pareto_data = {}
for scenario_name in ["Baseline", "Mid_future"]:
    if "pareto_dict" in globals() and scenario_name in pareto_dict:
        pareto_data[scenario_name] = pareto_dict[scenario_name].head(N_DISPLAY_PARETO).copy()
    elif scenario_name == "Baseline" and "pareto_base" in globals():
        pareto_data[scenario_name] = pareto_base.head(N_DISPLAY_PARETO).copy()
    elif scenario_name == "Mid_future" and "pareto_mid" in globals():
        pareto_data[scenario_name] = pareto_mid.head(N_DISPLAY_PARETO).copy()

# جفت‌اهداف مطابق ساختار جدول ۳
objective_pairs = [
    ("EUI", "OCI", "EUI–OCI"),
    ("EUI", "IDD", "EUI–IDD"),
    ("OCI", "IDD", "OCI–IDD")
]

scenarios = ["Baseline", "Mid_future"]
scenario_labels = {"Baseline": "Baseline scenario", "Mid_future": "Mid-future scenario"}

In [ ]:
# ---------------- 2. محاسبه شاخص‌های آماری ----------------
table_rows = []

for scen in scenarios:
    df_p = pareto_data[scen]

    for obj1, obj2, pair_label in objective_pairs:
        x = df_p[obj1].values.astype(float)
        y = df_p[obj2].values.astype(float)

        # محاسبه همبستگی‌ها و شیب خط رگرسیون
        r_p, _ = pearsonr(x, y)
        r_s, _ = spearmanr(x, y)
        slope, _, _, _, _ = linregress(x, y)

        table_rows.append({
            "Scenario": scenario_labels[scen],
            "Pair": pair_label,
            "Pearson (r)": f"{r_p:.3f}",
            "Spearman (ρ)": f"{r_s:.3f}",
            "Regression slope": f"{slope:.4f}"
        })

In [ ]:
# ---------------- 3. ساخت DataFrame و نمایش جدولی ----------------
table3_df = pd.DataFrame(table_rows)

# تنظیم نمایش مشابه جدول مقاله
table3_display = table3_df.copy()
table3_display.loc[table3_display["Scenario"].duplicated(), "Scenario"] = ""

print("\nTable 3. Pairwise relationships among Pareto-optimal objective values under baseline and mid-future scenarios.")
display(table3_display)

In [ ]:
# ---------------- 4. ذخیره خروجی‌ها ----------------
table3_df.to_csv("Table3_Pairwise_Relationships.csv", index=False)
table3_df.to_excel("Table3_Pairwise_Relationships.xlsx", index=False)

print("\nSaved files:")
print(" - Table3_Pairwise_Relationships.csv")
print(" - Table3_Pairwise_Relationships.xlsx")

In [ ]:
# ============ CELL 7: Zip and download all outputs ============
import glob, zipfile
from google.colab import files

# Collect result files, but leave out your 2 uploaded input CSVs
input_files = set(names) if "names" in globals() else set()
out_files = sorted(f for f in glob.glob("*.csv") + glob.glob("*.png") + glob.glob("*.pdf")
                   if f not in input_files)

with zipfile.ZipFile("results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in out_files:
        z.write(f)

print("Files in results.zip:")
for f in out_files:
    print("  -", f)

files.download("results.zip")